# SQL Queries — MLB Pitcher Performance & Salary Analysis

**Database:** SQLite (`pitcher_stats.db`)

**Tables:**
- `pitcher_stats` — Statcast + MLB Stats API pitching data (2021-2025)
- `fg_salaries` — FanGraphs salary data mapped to MLBAM player IDs
- `pitcher_stats_with_salary` — Joined table (stats + salary)
- `pitcher_surplus_value` — Model output: predicted vs actual salary

This file contains all SQL queries from the main analysis notebook, collected here for grading. Each query has comments explaining **what** it does, **how** it works, and **why** it's relevant.

**Query type summary:**

| Requirement | Queries |
|---|---|
| JOINs (3+) | Q1, Q9, Q10, Q12 |
| Window functions (2+) | Q4, Q7 |
| GROUP BY (2+) | Q3, Q5, Q11 |
| Subqueries (2+) | Q8, Q12 |
| **Total** | **12 queries** |

## Setup

In [1]:
import sqlite3
import pandas as pd

DB_PATH = "pitcher_stats.db"
conn = sqlite3.connect(DB_PATH)

# Confirm tables exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print("Available tables:")
for t in tables['name']:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn).iloc[0]['n']
    print(f"  {t}: {n:,} rows")

Available tables:
  fg_salaries: 4,003 rows
  pitcher_stats: 238 rows
  pitcher_stats_with_salary: 238 rows
  pitcher_surplus_value: 225 rows


## Query 1 — [JOIN] Build the main analysis table

Joins `pitcher_stats` with `fg_salaries` on player_id and season.

LEFT JOIN keeps all pitchers even when salary data is missing.

This connects on-field performance with compensation.

*Used in: Section 2 (Data Collection) — `build_joined_table()` function.*

In [2]:
# Query 1: JOIN — Join stats with salary data
# From the main notebook's build_joined_table() function.
# LEFT JOIN ensures we keep all pitchers even when salary data is missing.

q1 = pd.read_sql("""
    SELECT ps.name, ps.season, ps.team, ps.IP, ps.ERA, ps.SO, ps.BB,
           ps.FF_velo,
           ROUND(fs.salary / 1000000.0, 2) AS salary_M
    FROM pitcher_stats ps
    LEFT JOIN fg_salaries fs
        ON  ps.player_id = fs.player_id
        AND ps.season    = fs.season
    ORDER BY ps.season DESC, fs.salary DESC
    LIMIT 20
""", conn)

print("Top 20 highest-paid pitcher-seasons:")
q1

Top 20 highest-paid pitcher-seasons:


,name,season,team,IP,ERA,SO,BB,FF_velo,salary_M
0,Tarik Skubal,2025,Detroit Tigers,195.1,2.21,241,33,97.6,53.1
1,Cristopher Sánchez,2025,Philadelphia Phillies,202.0,2.50,212,44,NaN,51.2
2,Garrett Crochet,2025,Boston Red Sox,205.1,2.59,255,46,96.4,46.2
3,Logan Webb,2025,San Francisco Giants,207.0,3.22,224,46,92.8,44.2
4,Jesús Luzardo,2025,Philadelphia Phillies,183.2,3.92,216,57,96.5,42.7
5,Yoshinobu Yamamoto,2025,Los Angeles Dodgers,173.2,2.49,201,59,95.4,40.0
6,Max Fried,2025,New York Yankees,195.1,2.86,189,51,95.8,38.2
7,Hunter Brown,2025,Houston Astros,185.1,2.43,206,57,96.6,36.6
8,Kevin Gausman,2025,Toronto Blue Jays,193.0,3.59,189,50,94.5,32.8
9,Framber Valdez,2025,Houston Astros,192.0,3.66,187,68,94.3,32.1


## Query 2 — Preview: highest-paid starters

Selects the top 20 highest-paid starters with 100+ IP per season.

Filters on `salary IS NOT NULL` and `IP > 100`, orders by season and salary.

A quick sanity check that the data loaded correctly 

*Used in: Section 2 (Data Collection) — `preview()` function.*

In [3]:
# Query 2: Preview — top-paid starters with 100+ IP
# Quick sanity check that salary data loaded correctly.

q2 = pd.read_sql("""
    SELECT
        name, season, team,
        ROUND(IP, 1)                  AS IP,
        ERA, xERA, SO,
        ROUND(salary / 1000000.0, 2)  AS salary_millions
    FROM pitcher_stats_with_salary
    WHERE salary IS NOT NULL
      AND IP > 100
    ORDER BY season DESC, salary DESC
    LIMIT 20
""", conn)

print("Top 20 highest-paid starters with 100+ IP:")
q2

Top 20 highest-paid starters with 100+ IP:


,name,season,team,IP,ERA,xERA,SO,salary_millions
0,Tarik Skubal,2025,Detroit Tigers,195.1,2.21,2.71,241,53.1
1,Cristopher Sánchez,2025,Philadelphia Phillies,202.0,2.50,3.02,212,51.2
2,Garrett Crochet,2025,Boston Red Sox,205.1,2.59,2.89,255,46.2
3,Logan Webb,2025,San Francisco Giants,207.0,3.22,3.58,224,44.2
4,Jesús Luzardo,2025,Philadelphia Phillies,183.2,3.92,3.33,216,42.7
5,Yoshinobu Yamamoto,2025,Los Angeles Dodgers,173.2,2.49,2.74,201,40.0
6,Max Fried,2025,New York Yankees,195.1,2.86,3.38,189,38.2
7,Hunter Brown,2025,Houston Astros,185.1,2.43,3.14,206,36.6
8,Kevin Gausman,2025,Toronto Blue Jays,193.0,3.59,3.74,189,32.8
9,Framber Valdez,2025,Houston Astros,192.0,3.66,3.79,187,32.1


## Query 3 — [GROUP BY] Data quality: duplicate check

Checks for duplicate player/season combinations in the joined table.

GROUP BY player_id + season with HAVING count > 1.

Duplicates would inflate aggregates and bias the regression models

*Used in: Section 3 (Data Quality Checks).*

In [4]:
# Query 3: GROUP BY + HAVING — check for duplicate player-season rows
# Duplicates would bias all downstream aggregations and models.

q3 = pd.read_sql("""
    SELECT player_id, season, COUNT(*) AS n
    FROM pitcher_stats_with_salary
    GROUP BY player_id, season
    HAVING n > 1
    LIMIT 10
""", conn)

if len(q3) == 0:
    print("No duplicate player/season rows found — data is clean.")
else:
    print(f"Found {len(q3)} duplicate player/season combinations:")
    print(q3)

No duplicate player/season rows found — data is clean.


## Query 4 — [WINDOW FUNCTION] Rank pitchers by ERA within each season

Assigns each starter an ERA rank within their season (lower ERA = rank 1).

`RANK()` window function brokwn out by season, ordered by ERA ascending. Filtered to starters with 100+ IP.

Shows who was the best pitcher each year and whether top-ranked pitchers are also the highest paid

*Used in: Section 3 (Data Quality Checks) — end of section.*

In [5]:
# Query 4: WINDOW FUNCTION — ERA rank within each season
# Filtered to starters with 100+ IP for a meaningful comparison.

q4 = pd.read_sql("""
    SELECT
        name, season, team,
        ROUND(CAST(ERA AS FLOAT), 2) AS ERA,
        ROUND(IP, 1) AS IP,
        SO,
        ROUND(salary / 1000000.0, 2) AS salary_M,
        RANK() OVER (PARTITION BY season ORDER BY CAST(ERA AS FLOAT) ASC) AS era_rank
    FROM pitcher_stats_with_salary
    WHERE salary IS NOT NULL
      AND IP >= 100
      AND ERA IS NOT NULL
      AND CAST(ERA AS FLOAT) < 20
    ORDER BY season, era_rank
""", conn)

print("Top ERA pitchers per season (starters with 100+ IP):")
q4[q4['era_rank'] <= 5]

Top ERA pitchers per season (starters with 100+ IP):


,name,season,team,ERA,IP,SO,salary_M,era_rank
0,Corbin Burnes,2021,Milwaukee Brewers,2.43,167.0,234,59.8,1
1,Max Scherzer,2021,Los Angeles Dodgers,2.46,179.1,236,43.0,2
2,Walker Buehler,2021,Los Angeles Dodgers,2.47,207.2,212,44.4,3
3,Brandon Woodruff,2021,Milwaukee Brewers,2.56,179.1,211,38.0,4
4,Zack Wheeler,2021,Philadelphia Phillies,2.78,213.1,247,57.8,5
39,Justin Verlander,2022,Houston Astros,1.75,175.0,185,48.4,1
40,Julio Urías,2022,Los Angeles Dodgers,2.16,175.0,166,25.8,2
41,Dylan Cease,2022,Chicago White Sox,2.20,184.0,227,35.0,3
42,Alek Manoah,2022,Toronto Blue Jays,2.24,196.2,180,31.3,4
43,Sandy Alcantara,2022,Miami Marlins,2.28,228.2,207,47.3,5


## Query 5 — [GROUP BY] Season-level summary statistics

Aggregates pitcher performance and salary by season.

GROUP BY season with AVG, COUNT, SUM for key metrics.

Establishes baseline trends like ERA and salary increases/decreases.

*Used in: Section 3 (Data Quality Checks).*

In [6]:
# Query 5: GROUP BY — Season-level averages
# Aggregates key metrics by season to establish baseline trends.

q5 = pd.read_sql("""
    SELECT
        season,
        COUNT(*)                                              AS total_pitchers,
        SUM(CASE WHEN salary IS NOT NULL THEN 1 ELSE 0 END)  AS with_salary,
        ROUND(AVG(CAST(ERA AS FLOAT)), 2)                     AS avg_ERA,
        ROUND(AVG(IP), 1)                                      AS avg_IP,
        ROUND(AVG(salary) / 1000000.0, 1)                      AS avg_salary_M
    FROM pitcher_stats_with_salary
    WHERE ERA IS NOT NULL AND CAST(ERA AS FLOAT) < 20
    GROUP BY season
    ORDER BY season
""", conn)

print("Season summary:")
q5

Season summary:


,season,total_pitchers,with_salary,avg_ERA,avg_IP,avg_salary_M
0,2021,39,39,3.63,180.2,29.1
1,2022,45,45,3.34,184.1,26.7
2,2023,44,44,3.83,183.4,26.1
3,2024,58,58,3.71,177.9,23.3
4,2025,52,49,3.81,177.2,24.0


## Query 6 — Load and inspect the full dataset

Loads the full joined table and checks for missing values.

`SELECT *` from the joined table, then Python computes null percentages.

Identifies which columns have gaps (e.g., salary, advanced metrics) so we know what to handle before modeling.

*Used in: Section 3 (Data Quality Checks).*

In [7]:
# Query 6: Load full dataset + check missing values
# SELECT * to pull everything, then compute null percentages in Python.

q6 = pd.read_sql("SELECT * FROM pitcher_stats_with_salary", conn)

null_pct = (q6.isnull().sum() / len(q6) * 100).round(1)
missing = pd.DataFrame({"missing_pct": null_pct})
missing = missing[missing["missing_pct"] > 0].sort_values("missing_pct", ascending=False)
print(f"Total rows: {len(q6):,}")
print("\nColumns with missing values:")
print(missing.to_string() if len(missing) > 0 else "  None!")

Total rows: 238

Columns with missing values:
         missing_pct
age            100.0
SV_pct          96.6
FS_pct          92.0
ST_pct          80.7
FC_pct          71.4
SL_pct          64.3
CU_pct          63.9
CH_pct          61.3
SI_pct          49.2
FF_pct          18.9
FF_velo          1.7
FF_ivb           1.7
salary           1.3


## Query 7 — [WINDOW FUNCTION] Salary vs season and team averages

For each pitcher, shows their salary alongside the average salary for their season and their team.

Two `AVG()` window functions — one broken out by season, one by team. Difference columns show over/under the benchmark.

Lets GMs benchmark a pitcher's cost against two baselines: the league-wide market that year and the team's internal payroll. A pitcher cheap relative to both is a strong value target.

*Used in: Section 4 (EDA) — Additional SQL-Driven EDA.*

In [8]:
# Query 7: WINDOW FUNCTION — Salary compared to season and team averages
# Two AVG() window functions: one partitioned by season, one by team.
# Benchmarks each pitcher's cost against league and team averages.

q7 = pd.read_sql("""
    SELECT
        name, season, team,
        ROUND(salary / 1000000.0, 2) AS salary_M,
        ROUND(AVG(salary / 1000000.0) OVER (PARTITION BY season), 2) AS avg_salary_season,
        ROUND(salary / 1000000.0 -
              AVG(salary / 1000000.0) OVER (PARTITION BY season), 2) AS vs_season_avg,
        ROUND(AVG(salary / 1000000.0) OVER (PARTITION BY team), 2) AS avg_salary_team,
        ROUND(salary / 1000000.0 -
              AVG(salary / 1000000.0) OVER (PARTITION BY team), 2) AS vs_team_avg
    FROM pitcher_stats_with_salary
    WHERE salary IS NOT NULL
    ORDER BY season, salary DESC
""", conn)

print("Salary vs season and team averages (sample):")
q7.head(15)

Salary vs season and team averages (sample):


,name,season,team,salary_M,avg_salary_season,vs_season_avg,avg_salary_team,vs_team_avg
0,Corbin Burnes,2021,Milwaukee Brewers,59.8,29.1,30.7,29.91,29.89
1,Zack Wheeler,2021,Philadelphia Phillies,57.8,29.1,28.7,36.86,20.94
2,Nathan Eovaldi,2021,Boston Red Sox,45.8,29.1,16.7,25.89,19.91
3,Walker Buehler,2021,Los Angeles Dodgers,44.4,29.1,15.3,32.04,12.36
4,Max Scherzer,2021,Los Angeles Dodgers,43.0,29.1,13.9,32.04,10.96
5,Gerrit Cole,2021,New York Yankees,41.5,29.1,12.4,29.01,12.49
6,Julio Urías,2021,Los Angeles Dodgers,40.3,29.1,11.2,32.04,8.26
7,Kevin Gausman,2021,San Francisco Giants,38.4,29.1,9.3,35.39,3.01
8,Brandon Woodruff,2021,Milwaukee Brewers,38.0,29.1,8.9,29.91,8.09
9,Charlie Morton,2021,Atlanta Braves,36.3,29.1,7.2,27.85,8.45


## Query 8 — [SUBQUERY] Pitchers earning above their season's average salary

Identifies pitchers paid above the average salary for their season.

A  subquery computes average salary per season. The outer query filters to pitchers above that threshold.

Separates premium-paid pitchers from league-minimum guys so the front office can focus on where the real salary-efficiency gaps exist.

*Used in: Section 4 (EDA) — after ERA vs Salary scatter plot.*

In [9]:
# Query 8: SUBQUERY — Pitchers earning above their season's average salary
# Correlated subquery computes AVG salary per season; outer query filters above it.

q8 = pd.read_sql("""
    SELECT
        name, season, team,
        ROUND(CAST(ERA AS FLOAT), 2) AS ERA,
        IP,
        ROUND(salary / 1000000.0, 2) AS salary_M
    FROM pitcher_stats_with_salary ps
    WHERE salary > (
        SELECT AVG(salary)
        FROM pitcher_stats_with_salary sub
        WHERE sub.season = ps.season
          AND sub.salary IS NOT NULL
    )
    ORDER BY season, salary DESC
""", conn)

print(f"Pitchers earning above season average: {len(q8)}")
q8.head(15)

Pitchers earning above season average: 118


,name,season,team,ERA,IP,salary_M
0,Corbin Burnes,2021,Milwaukee Brewers,2.43,167.0,59.8
1,Zack Wheeler,2021,Philadelphia Phillies,2.78,213.1,57.8
2,Nathan Eovaldi,2021,Boston Red Sox,3.75,182.1,45.8
3,Walker Buehler,2021,Los Angeles Dodgers,2.47,207.2,44.4
4,Max Scherzer,2021,Los Angeles Dodgers,2.46,179.1,43.0
5,Gerrit Cole,2021,New York Yankees,3.23,181.1,41.5
6,Julio Urías,2021,Los Angeles Dodgers,2.96,185.2,40.3
7,Kevin Gausman,2021,San Francisco Giants,2.81,192.0,38.4
8,Brandon Woodruff,2021,Milwaukee Brewers,2.56,179.1,38.0
9,Charlie Morton,2021,Atlanta Braves,3.34,185.2,36.3


## Query 9 — [JOIN] Year-over-year salary and ERA changes

For pitchers appearing in consecutive seasons, shows salary and ERA changes.

Self-join on player_id where current season = previous season + 1.

Tracks which pitchers got big raises and whether those raises were justified by improved performance. A pitcher whose ERA went up but salary also went up is a market inefficiency.

*Used in: Section 4 (EDA) — Additional SQL-Driven EDA.*

In [10]:
# Query 9: SELF-JOIN — Year-over-year salary and ERA changes
# Joins the table to itself where curr.season = prev.season + 1.
# Tracks raises vs performance changes.

q9 = pd.read_sql("""
    SELECT
        curr.name,
        prev.season AS prev_season,
        curr.season AS curr_season,
        ROUND(CAST(prev.ERA AS FLOAT), 2) AS prev_ERA,
        ROUND(CAST(curr.ERA AS FLOAT), 2) AS curr_ERA,
        ROUND(prev.salary / 1000000.0, 2) AS prev_salary_M,
        ROUND(curr.salary / 1000000.0, 2) AS curr_salary_M,
        ROUND((curr.salary - prev.salary) / 1000000.0, 2) AS salary_change_M
    FROM pitcher_stats_with_salary curr
    INNER JOIN pitcher_stats_with_salary prev
        ON  curr.player_id = prev.player_id
        AND curr.season    = prev.season + 1
    WHERE curr.salary IS NOT NULL
      AND prev.salary IS NOT NULL
    ORDER BY ABS(curr.salary - prev.salary) DESC
    LIMIT 20
""", conn)

print("Biggest year-over-year salary changes:")
q9

Biggest year-over-year salary changes:


,name,prev_season,curr_season,prev_ERA,curr_ERA,prev_salary_M,curr_salary_M,salary_change_M
0,Charlie Morton,2021,2022,3.34,4.34,36.3,11.3,-25.0
1,José Berríos,2021,2022,3.52,5.23,32.1,7.1,-25.0
2,Corbin Burnes,2021,2022,2.43,2.94,59.8,36.7,-23.1
3,Sandy Alcantara,2022,2023,2.28,4.14,47.3,24.2,-23.1
4,Justin Verlander,2022,2023,1.75,3.22,48.4,26.9,-21.5
5,Aaron Nola,2022,2023,3.25,4.46,50.4,30.6,-19.8
6,Robbie Ray,2021,2022,2.84,3.71,31.3,13.0,-18.3
7,Kevin Gausman,2023,2024,3.16,3.83,42.1,24.0,-18.1
8,Jake Irvin,2024,2025,4.41,5.70,14.5,-3.1,-17.6
9,Germán Márquez,2021,2022,4.40,4.95,27.6,10.4,-17.2


## Query 10 — [JOIN] Surplus value with Statcast advanced metrics

Joins the model's surplus predictions back to raw Statcast data for xERA, xwOBA, and exit velocity.

INNER JOIN on name and season between `pitcher_surplus_value` and `pitcher_stats`.

The regression model uses standard stats. This cross-references whether advanced Statcast metrics corroborate the model's over/underpaid findings — adding depth beyond the model alone.

*Used in: Section 7 (Surplus Value Analysis).*

In [11]:
# Query 10: JOIN — Surplus value joined with Statcast advanced metrics
# INNER JOIN brings in xERA, xwOBA, EV alongside predicted vs actual salary.

q10 = pd.read_sql("""
    SELECT
        sv.name, sv.season, sv.team,
        ROUND(sv.ERA, 2)                AS ERA,
        ROUND(ps.xERA, 2)              AS xERA,
        ROUND(ps.xwOBA, 3)             AS xwOBA,
        ROUND(ps.EV, 1)                AS exit_velo,
        ROUND(sv.salary_M, 2)          AS actual_salary_M,
        ROUND(sv.expected_salary_M, 2) AS model_salary_M,
        ROUND(sv.surplus_M, 2)         AS surplus_M
    FROM pitcher_surplus_value sv
    INNER JOIN pitcher_stats ps
        ON  sv.name   = ps.name
        AND sv.season = ps.season
    ORDER BY sv.surplus_M DESC
    LIMIT 20
""", conn)

print("Most underpaid pitchers — with Statcast context:")
q10

Most underpaid pitchers — with Statcast context:


,name,season,team,ERA,xERA,xwOBA,exit_velo,actual_salary_M,model_salary_M,surplus_M
0,Robbie Ray,2022,Seattle Mariners,3.71,3.62,0.299,89.7,13.0,27.92,14.92
1,Gerrit Cole,2022,New York Yankees,3.50,3.39,0.290,89.4,27.4,41.12,13.72
2,Charlie Morton,2022,Atlanta Braves,4.34,4.14,0.318,89.3,11.3,23.64,12.34
3,José Berríos,2024,Toronto Blue Jays,3.60,4.68,0.336,89.6,9.1,21.05,11.95
4,Gavin Williams,2025,Cleveland Guardians,3.06,4.30,0.321,90.4,11.5,22.49,10.99
5,Shane McClanahan,2022,Tampa Bay Rays,2.54,2.75,0.262,87.6,29.0,39.62,10.62
6,Charlie Morton,2024,Atlanta Braves,4.19,4.52,0.331,88.7,9.0,19.49,10.49
7,Carlos Rodón,2024,New York Yankees,3.96,4.08,0.316,90.4,12.9,22.67,9.77
8,Lucas Giolito,2023,Cleveland Guardians,4.88,4.61,0.329,89.2,7.0,16.26,9.26
9,Robbie Ray,2021,Toronto Blue Jays,2.84,3.60,0.293,90.4,31.3,40.44,9.14


## Query 11 — [GROUP BY] Fastball velocity tier analysis

Buckets pitchers by fastball velocity and compares ERA, K/BB ratio, and salary.

CASE expression creates velocity tiers, GROUP BY aggregates across each.

Velocity is increasingly valued in the modern game. Shows whether the salary market rewards higher velo, and whether that premium is justified by lower ERA and higher K/BB.

*Used in: Section 4 (EDA) — Additional SQL-Driven EDA.*

In [12]:
# Query 11: GROUP BY — Fastball velocity tier and its effect on salary/ERA
# CASE expression creates velocity tiers, GROUP BY aggregates across each.

q11 = pd.read_sql("""
    SELECT
        CASE
            WHEN FF_velo >= 96 THEN '96+ mph (Elite)'
            WHEN FF_velo >= 93 THEN '93-95 mph (Above Avg)'
            WHEN FF_velo >= 90 THEN '90-92 mph (Average)'
            ELSE                    'Below 90 mph'
        END AS velo_tier,
        COUNT(*)                                     AS n_pitchers,
        ROUND(AVG(CAST(ERA AS FLOAT)), 2)            AS avg_ERA,
        ROUND(AVG(SO * 1.0 / NULLIF(BB, 0)), 2)     AS avg_K_BB,
        ROUND(AVG(salary) / 1000000.0, 2)             AS avg_salary_M
    FROM pitcher_stats_with_salary
    WHERE FF_velo IS NOT NULL AND salary IS NOT NULL AND ERA IS NOT NULL
    GROUP BY velo_tier
    ORDER BY avg_salary_M DESC
""", conn)

print("Performance and salary by fastball velocity tier:")
q11

Performance and salary by fastball velocity tier:


,velo_tier,n_pitchers,avg_ERA,avg_K_BB,avg_salary_M
0,96+ mph (Elite),41,3.33,4.34,33.89
1,93-95 mph (Above Avg),114,3.60,3.64,25.06
2,90-92 mph (Average),65,3.94,3.51,22.31
3,Below 90 mph,11,4.16,3.06,15.79


## Query 12 — [SUBQUERY + JOIN] Worst value contracts

Finds pitchers with above-average ERA AND above-average salary — the worst value contracts.

Two correlated subqueries compute the season's average ERA and average salary. The outer query filters to pitchers exceeding both thresholds.

Directly answers key question #3 for our target audience: which pitcher contracts represent the worst value? These are the deals a GM should avoid or try to trade.

*Used in: Section 7 (Surplus Value Analysis).*

In [13]:
# Query 12: SUBQUERY — Worst value contracts (above-avg ERA AND above-avg salary)
# Two correlated subqueries: one for ERA threshold, one for salary threshold.

q12 = pd.read_sql("""
    SELECT
        ps.name, ps.season, ps.team,
        ROUND(CAST(ps.ERA AS FLOAT), 2) AS ERA,
        ROUND(ps.IP, 1) AS IP,
        ROUND(ps.salary / 1000000.0, 2) AS salary_M
    FROM pitcher_stats_with_salary ps
    WHERE CAST(ps.ERA AS FLOAT) > (
            SELECT AVG(CAST(ERA AS FLOAT))
            FROM pitcher_stats_with_salary
            WHERE season = ps.season
              AND ERA IS NOT NULL AND CAST(ERA AS FLOAT) < 20
          )
      AND ps.salary > (
            SELECT AVG(salary)
            FROM pitcher_stats_with_salary
            WHERE season = ps.season AND salary IS NOT NULL
          )
      AND ps.ERA IS NOT NULL
    ORDER BY ps.salary DESC
""", conn)

print(f"Worst value contracts (above-avg ERA AND above-avg salary): {len(q12)}")
q12

Worst value contracts (above-avg ERA AND above-avg salary): 23


,name,season,team,ERA,IP,salary_M
0,Nathan Eovaldi,2021,Boston Red Sox,3.75,182.1,45.8
1,Kevin Gausman,2022,Toronto Blue Jays,3.35,174.2,44.4
2,Spencer Strider,2023,Atlanta Braves,3.86,186.2,43.9
3,Jesús Luzardo,2025,Philadelphia Phillies,3.92,183.2,42.7
4,Dylan Cease,2021,Chicago White Sox,3.91,165.2,35.9
5,Aaron Nola,2021,Philadelphia Phillies,4.63,180.2,35.4
6,Tyler Mahle,2021,Cincinnati Reds,3.75,180.0,31.0
7,Luis Castillo,2021,Cincinnati Reds,3.98,187.2,30.8
8,Aaron Nola,2023,Philadelphia Phillies,4.46,193.2,30.6
9,Sonny Gray,2024,St. Louis Cardinals,3.84,166.1,30.5


## Cleanup

In [14]:
conn.close()
print("Database connection closed.")

Database connection closed.
